<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Cuaderno, el Casillero y el Mapa de Amigos: Dónde Guardar los Datos 🗃️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 01
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/Para%20Dummies/00_Opciones_Procesamiento_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 01 (Opciones de procesamiento)](../00_Opciones_de_Procesamiento_Hadoop_Spark_NoSQL_SQL.ipynb). Cuando los datos crecen, hay que decidir **dónde guardarlos** y **cuándo procesarlos**. No hay una respuesta única, igual que en una tienda de barrio no usas **lo mismo** para todo:

| En la tienda de don José | En el mundo de los datos |
|---|---|
| El **cuaderno de cuentas** con columnas fijas (cliente, producto, valor) | Base **relacional** (tablas y SQL) |
| Los **casilleros del guardarropa** (te dan una ficha con número y sacas tu abrigo) | **Clave-valor**: una clave, un valor |
| Una **carpeta por cada productor** (cada una con hojas distintas) | **Documentos**: cada ficha puede tener campos distintos |
| El **mapa de quién conoce a quién** en el pueblo | **Grafo**: personas y conexiones |
| El **cierre de caja del sábado** frente a la **caja registradora que suma al instante** | Procesamiento **por lotes** frente a procesamiento **en flujo** |

Al terminar podrás explicar:
1. La diferencia entre procesar **por lotes** y **en flujo** (¿cuándo sé el total?).
2. Cuándo conviene un **casillero** (clave-valor), una **carpeta** (documentos) o un **mapa** (grafo).
3. Por qué, cuando **se corta la línea** entre dos sucursales, hay que elegir entre **seguridad** (no vender) y **servicio** (vender igual y arriesgarse).

> 🔗 Si luego quieres ver las versiones **desde cero y con medidas reales** (SQLite, DuckDB, los cuatro almacenes NoSQL, la simulación de la partición de red y Spark local): [cuaderno estándar](../00_Opciones_de_Procesamiento_Hadoop_Spark_NoSQL_SQL.ipynb). Antes de esto: [el ciclo de vida de los datos](../../00%20-%20Fundamentos%20de%20Big%20Data/Para%20Dummies/03_Ciclo_de_Vida_Dummies.ipynb).

---
## 1. El Cierre de Caja o la Caja que Suma al Instante 🧾

Una tienda anota cada venta con su hora. Hay dos formas de saber **cuánto se ha vendido**:

- **Por lotes (cierre de caja):** se espera hasta las **12:00**, se suman **todas** las ventas de la mañana y recién ahí se sabe el total. *Respuesta exacta, pero tarde.*
- **En flujo (caja registradora):** **cada venta suma** al instante a un total que siempre está a la vista. *Respuesta inmediata.*

A las **11:00**, ¿cuánto llevamos? Veamos qué contesta cada forma. (Las ventas son **inventadas**.)

In [ ]:
import pandas as pd

ventas = [("09:05", 12000), ("09:40", 8000), ("10:15", 15000), ("10:50", 5000), ("11:20", 20000), ("11:45", 7000)]   # (hora, monto): datos inventados


def minutos(hora):
    return int(hora[:2]) * 60 + int(hora[3:])


def respuesta_por_lotes(hora_consulta, cierre="12:00"):
    """El lote se cierra a las 12:00: antes de eso NO hay total (None)."""
    if minutos(hora_consulta) < minutos(cierre):
        return None
    return sum(m for _, m in ventas)


def respuesta_en_flujo(hora_consulta):
    """La caja registradora siempre tiene el acumulado hasta ese momento."""
    return sum(m for h, m in ventas if minutos(h) <= minutos(hora_consulta))


for consulta in ["10:00", "11:00", "12:00"]:
    print(f"A las {consulta}: por lotes = {respuesta_por_lotes(consulta)} | en flujo = {respuesta_en_flujo(consulta)}")

assert respuesta_por_lotes("11:00") is None and respuesta_en_flujo("11:00") == 12000 + 8000 + 15000 + 5000          # en flujo: 40 000
assert respuesta_por_lotes("12:00") == respuesta_en_flujo("12:00") == 67000                                          # al cierre, ¡coinciden!

---
**Lección:** a las **11:00** el cierre de caja **todavía no puede responder** (`None`), mientras que la caja registradora ya sabe que van **$40 000**. A las **12:00** **las dos coinciden: $67 000**. Es la gran idea: **el resultado final es el mismo**; lo que cambia es **cuándo lo sabes**. Si decides con el total del mes (la factura), el lote sirve; si necesitas una **alerta** (¿alguien está usando mi tarjeta *ahora*?), necesitas flujo.

---

## 2. Casilleros, Carpetas y el Mapa de Amigos 🗂️

Tres formas de guardar que **no son una tabla**:

1. **Casillero (clave-valor):** me das tu **ficha** (la clave) y te entrego tu abrigo (el valor), al instante. Además, **el ticket del parqueadero vence**: pasado el tiempo, el casillero se libera.
2. **Carpetas (documentos):** cada productor tiene **su carpeta**, y **no todas tienen las mismas hojas** (el de papa anota la variedad; el de fresa anota si tiene invernadero). Se pueden buscar carpetas por lo que dicen.
3. **Mapa de amigos (grafo):** quién conoce a quién. Preguntar **«los amigos de mis amigos que aún no conozco»** es natural en un mapa y **incómodo** en una tabla.

In [ ]:
# --- 1. Casillero con ticket que vence (el "reloj" es un contador de minutos) ---
ahora = 0
casilleros = {}                                                                 # ficha -> (abrigo, minuto en que vence o None)


def guardar(ficha, abrigo, vigente_min=None):
    casilleros[ficha] = (abrigo, None if vigente_min is None else ahora + vigente_min)


def sacar(ficha):
    if ficha not in casilleros:
        return None
    abrigo, vence = casilleros[ficha]
    if vence is not None and ahora >= vence:
        del casilleros[ficha]                                                   # el ticket venció: el casillero se libera
        return None
    return abrigo


guardar("ficha-17", "abrigo azul", vigente_min=30)                              # parqueadero: 30 minutos
guardar("ficha-18", "ruana")                                                    # sin vencimiento
ahora = 20
assert sacar("ficha-17") == "abrigo azul"                                       # a los 20 min, todavía vigente
ahora = 45
assert sacar("ficha-17") is None and sacar("ficha-18") == "ruana"               # a los 45 min, el ticket venció; la ruana sigue
print("Casillero: a los 20 min sale el abrigo; a los 45 min el ticket venció, pero la ruana sigue guardada.")

# --- 2. Carpetas: cada una con hojas distintas ---
carpetas = [
    {"productor": "Rosa", "producto": "papa", "municipio": "Tunja", "variedad": "pastusa", "precio_kg": 1800},
    {"productor": "Luis", "producto": "papa", "municipio": "Duitama", "variedad": "criolla", "precio_kg": 2600},
    {"productor": "Marta", "producto": "fresa", "municipio": "Tunja", "invernadero": True, "precio_kg": 5200},
    {"productor": "Pedro", "producto": "cebolla", "municipio": "Sogamoso", "precio_kg": 1500},
    {"productor": "Inés", "producto": "papa", "municipio": "Tunja", "variedad": "capiro", "precio_kg": 1950},
]
papa_en_tunja = [c["productor"] for c in carpetas if c["producto"] == "papa" and c["municipio"] == "Tunja"]
con_variedad = [c["productor"] for c in carpetas if "variedad" in c]            # solo algunas carpetas tienen esa hoja
print("Productores de papa en Tunja:", papa_en_tunja, "| carpetas con la hoja 'variedad':", con_variedad)
assert papa_en_tunja == ["Rosa", "Inés"] and con_variedad == ["Rosa", "Luis", "Inés"]

# --- 3. Mapa de amigos ---
amigos = {"Ana": {"Beto", "Carla"}, "Beto": {"Ana", "Diego"}, "Carla": {"Ana", "Diego", "Eva"}, "Diego": {"Beto", "Carla"}, "Eva": {"Carla"}}
def amigos_de_amigos(persona):
    candidatos = set()
    for amigo in amigos[persona]:
        candidatos |= amigos[amigo]
    return candidatos - amigos[persona] - {persona}                             # quito a mis amigos y a mi mismo


print("Amigos de amigos de Ana:", sorted(amigos_de_amigos("Ana")))
assert amigos_de_amigos("Ana") == {"Diego", "Eva"}

---
##### 🛠️ Práctica 1: ¿Quién vende papa barata?

Con la lista `carpetas` (los cinco productores de arriba):

1. Guarda en `baratos` la **lista de nombres de productores** (campo `"productor"`) que venden **papa** (`producto == "papa"`) a **menos de 2 000** pesos el kilo (`precio_kg < 2000`), en el orden en que aparecen.
2. Guarda en `mas_barato` el nombre del productor de **cualquier producto** con el `precio_kg` más bajo (pista: `min(carpetas, key=lambda c: c["precio_kg"])`).
3. Comprueba con `assert` que `baratos == ["Rosa", "Inés"]` y que `mas_barato == "Pedro"`.

In [ ]:
# Escribe tu código aquí

# baratos = ...
# mas_barato = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
baratos = [c["productor"] for c in carpetas if c["producto"] == "papa" and c["precio_kg"] < 2000]
mas_barato = min(carpetas, key=lambda c: c["precio_kg"])["productor"]
print("Papa por menos de $2 000:", baratos, "| el precio mas bajo de todos lo tiene:", mas_barato)

assert baratos == ["Rosa", "Inés"]
assert mas_barato == "Pedro"
```
</details>

---


---
## 3. Cuando se Corta la Línea entre Dos Sucursales ✂️📞

Don José tiene **dos tiendas** (Norte y Sur) que comparten **la misma cuenta de inventario**: de un producto quedan **5 unidades en total**, y cada tienda tiene **una copia** de esa cifra. Todo va bien mientras el teléfono entre las tiendas funciona. Pero **un día se corta la línea** y llegan **6 clientes** que quieren **1 unidad cada uno** (alternando Norte, Sur, Norte, Sur...). Cada tienda debe decidir:

- **Opción prudente:** «como no puedo confirmar con la otra tienda, **no vendo**». *Cero errores, pero clientes sin servicio.*
- **Opción rápida:** «vendo con lo que dice **mi copia**». *Todos atendidos, pero podemos vender lo que no tenemos.*

Esa es la esencia del famoso **teorema CAP**: con la línea cortada **no se puede tener a la vez** respuestas **siempre disponibles** y datos **siempre correctos**.

In [ ]:
STOCK_REAL = 5                                                                  # unidades que de verdad existen
clientes = ["Norte", "Sur", "Norte", "Sur", "Norte", "Sur"]                     # a que tienda llega cada cliente (6 clientes, 1 unidad cada uno)


def atender(modo, clientes):
    copia = {"Norte": STOCK_REAL, "Sur": STOCK_REAL}                            # cada tienda ve 5 unidades en SU copia
    vendidas = rechazados = 0
    for tienda in clientes:
        if modo == "prudente":
            rechazados += 1                                                     # sin confirmar con la otra tienda: no vende
        elif copia[tienda] > 0:
            copia[tienda] -= 1; vendidas += 1                                   # vende segun su copia local
        else:
            rechazados += 1
    sobreventa = max(0, vendidas - STOCK_REAL)
    return {"modo": modo, "vendidas": vendidas, "rechazados": rechazados, "sobreventa": sobreventa}


prudente, rapida = atender("prudente", clientes), atender("rapida", clientes)
print(pd.DataFrame([prudente, rapida]).set_index("modo").to_string())
assert prudente == {"modo": "prudente", "vendidas": 0, "rechazados": 6, "sobreventa": 0}
assert rapida["vendidas"] == 6 and rapida["sobreventa"] == 1 and rapida["rechazados"] == 0
print(f"\nPrudente: nadie recibe un producto inexistente, pero se pierden {prudente['rechazados']} ventas. Rápida: se vende todo, pero {rapida['sobreventa']} cliente se queda esperando algo que no existe (habrá que pedir disculpas o conseguirlo).")

---
**Lección:** con la línea cortada, la opción **prudente** no vendió nada (0 de 6) pero **tampoco prometió nada falso**; la **rápida** atendió a los 6, pero **vendió 1 unidad que no existía**. **Ninguna es «la correcta»**: depende del negocio. Para el **saldo de una cuenta bancaria** conviene ser prudente; para **«personas viendo este vuelo»** o los «me gusta» de una publicación, conviene ser rápido y **reconciliar después**.

---

## Resumen en una frase 🎯

> **No existe una sola forma de guardar y procesar datos: se elige según la pregunta (¿lo necesito ya o al cierre?, ¿busco por ficha, por contenido o por conexiones?) y, cuando la red falla, según lo que el negocio no puede perder: la exactitud o el servicio.**

### Lo que aprendiste hoy:
- ✅ **Lote o flujo:** el resultado final es el mismo; cambia **cuándo** lo sabes (a las 11:00 el lote aún no responde; el flujo ya lleva $40 000).
- ✅ **Casillero, carpetas y mapa:** clave-valor (con vencimiento), documentos (hojas distintas por carpeta) y grafo (amigos de amigos) son **herramientas distintas** para **preguntas distintas**.
- ✅ **La línea cortada:** prudente = 0 ventas y 0 errores; rápida = 6 ventas y 1 sobreventa; **elegir** es decisión de negocio.

> 🎓 **Siguiente paso:** [el cuaderno estándar](../00_Opciones_de_Procesamiento_Hadoop_Spark_NoSQL_SQL.ipynb) implementa los cuatro almacenes NoSQL desde cero, simula la partición de red con tres réplicas y mide el mismo cálculo en *pandas*, DuckDB, SQLite y Spark. Luego, [contar votos en varias mesas (MapReduce)](01_MapReduce_Dummies.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué un **reporte mensual** puede hacerse por lotes pero una **alerta de fraude** no? ¿Qué pierdes si usas flujo para todo?
2. Piensa en tu **teléfono**: ¿qué guardaría como **casillero** (ficha → valor), qué como **carpeta** y qué como **mapa**? Da un ejemplo de cada uno.
3. Un banco y una red social, ambos con dos sucursales y la línea cortada: ¿quién debería ser **prudente** y quién **rápido**? Explica por qué.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>